In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("womens_clothing_ecommerce_reviews.csv")
print(df.shape)
print(df.head())
print(df.isnull().sum())
print(df.duplicated().sum())
print("empty review text:",int((df['Review Text'].fillna("").str.strip() == "").sum()))
print("\nClass counts (imbalance matters for accuracy):")
#check class imbalance we have formulas for it 
print(df["sentiment"].value_counts())
print("\nClass share:")
print(df["sentiment"].value_counts(normalize=True).round(4))
print("\nLable values:",sorted(df["sentiment"].unique().tolist()))


In [ ]:
#duplicate full rows are not independent observation ; drop before the split
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n_before-len(df)} duplicate rows. Remaining: {len(df)}")

In [ ]:
x = df["Review Text"]
y = df["sentiment"]

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print("Train size:",len(x_train),"| Test size:",len(x_test))

In [ ]:
print("\nConverting text to numerical features using bag-of-word...")
vectorizer = CountVectorizer(stop_words='english')
x_train_bow = vectorizer.fit_transform(x_train)
print(f"x_train_bow shape:\n {x_train_bow.shape}")
print(f"Unique words (features): {vectorizer.get_feature_names_out()}")
print(x_train_bow)
#when we use test data but we dont fit it because it can be add new features then we dont fit it
x_test_bow = vectorizer.transform(x_test)


In [ ]:
#max iter mean tune the hyper parameters 
model = LogisticRegression(max_iter=1000,solver='lbfgs',random_state=42)
model.fit(x_train_bow,y_train)#bag-of-words
print("model training is complete")


In [ ]:
# accuracy = no.of.accuracy predictions / total predictions
y_test_pred = model.predict(x_test_bow)
y_train_pred = model.predict(x_train_bow)

accuracy = accuracy_score(y_train,y_train_pred)
print(f"Model Accuracy (train): {accuracy:.4f} ({accuracy:.2%})")


accuracy = accuracy_score(y_test,y_test_pred)
print(f"Model Accuracy (test): {accuracy:.4f} ({accuracy:.2%})")
print(y_test_pred)

In [ ]:
print(classification_report(y_train,y_train_pred))
print("-"*50)
print(classification_report(y_test,y_test_pred))

In [ ]:
print(confusion_matrix(y_test,y_test_pred))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,y_test_pred).plot()

inference


In [ ]:
review = pd.DataFrame({
    'Review_Text':["Really comfortable T-shirt. The fabric is soft and breathable, and the fit is perfect. The color also looks exactly like the pictures. Very happy with the quality",
                   "love these jeans! They fit perfectly and are very comfortable to wear. The denim feels strong and the color is great. Definitely worth the price.",
                   "The hoodie is very soft and comfortable. The size was exactly right and the material feels good quality. I have already worn it several times and it still looks new",
                   "The shirt looks nice, but the material is thinner than I expected. The size also feels smaller than the size chart. I recommend ordering one size larger.",
                   "The jeans look good, but they are too tight around the waist. The fabric is also a little stiff. I expected a more comfortable fit for this price.",
                   "The design is nice, but the quality was disappointing. Some stitching was loose when I received it, and the material feels cheaper than it looks in the photos."]
})

x_rev_bow = vectorizer.transform(review["Review_Text"])

predict = model.predict(x_rev_bow)
print(predict)
print(review.shape)